In [1]:
# ==============================================================================
# PIPELINE 2 (P2) — ATTENTION ALIGNMENT
# P1 + spatial attention distillation:
#   CNN layer4 channel-mean → pseudo-attention map
#   KL divergence against ViT CLS-to-patch attention distribution
# All 4 scales run automatically: 10% → 25% → 50% → 100%
# ==============================================================================

import os, random, math
import numpy as np
import torch, torch.nn as nn, torch.optim as optim, torch.nn.functional as F
import torchvision, torchvision.transforms as transforms
import timm

SEED            = 67
BATCH_SIZE      = 64
EPOCHS          = 10
LEARNING_RATE   = 8e-4
LABEL_SMOOTHING = 0.1
DROP_PATH_RATE  = 0.1
KD_TEMPERATURE  = 4.5
DECAY_RATE      = 0.10
TASK_WEIGHT     = 0.3
DISTILL_WEIGHT  = 0.7
ATTN_WEIGHT     = 0.5   # initial weight for attention alignment, also annealed
DEVICE          = torch.device("cuda" if torch.cuda.is_available() else "cpu")
TRAIN_DIR       = '/kaggle/input/datasets/melikechan/cifar100/cifar100/train'
TEST_DIR        = '/kaggle/input/datasets/melikechan/cifar100/cifar100/test'
MODEL_PATH      = '/kaggle/input/models/totallyapoorv/resnetoncifar100/pytorch/default/1/resnetall.pth'
DATA_SCALES     = ['10%', '25%', '50%', '100%']
SCALE_MAP       = {'10%': 0.10, '25%': 0.25, '50%': 0.50, '100%': 1.0}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.deterministic = True

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])
transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761)),
])

full_trainset = torchvision.datasets.ImageFolder(TRAIN_DIR, transform=transform_train)
testset       = torchvision.datasets.ImageFolder(TEST_DIR,  transform=transform_test)
all_indices   = list(range(len(full_trainset)))
random.shuffle(all_indices)

criterion_kl = nn.KLDivLoss(reduction='batchmean')

def soft_kd_loss(s_logits, t_logits, T=KD_TEMPERATURE):
    return criterion_kl(F.log_softmax(s_logits/T, dim=1),
                        F.softmax(t_logits/T, dim=1)) * (T**2)

def cosine_sem_loss(t_feat, s_feat, proj):
    return 1.0 - F.cosine_similarity(t_feat, proj(s_feat), dim=1).mean()

def attention_alignment_loss(teacher_spatial, qkv_tensor, attn_mod, B, T=1.0):
    # Reconstruct softmax attention from cached QKV — no model forward needed
    N   = qkv_tensor.size(1)
    qkv = qkv_tensor.reshape(B, N, 3, attn_mod.num_heads, attn_mod.head_dim)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, _ = qkv.unbind(0)
    attn = (q * attn_mod.scale) @ k.transpose(-2, -1)
    attn = attn.softmax(dim=-1)                          # [B, heads, N, N]

    # ViT: CLS token (idx 0) attending to patch tokens (idx 2 onward, skip DIST)
    vit_attn = attn[:, :, 0, 2:].mean(dim=1)            # [B, 196]
    vit_attn = F.softmax(vit_attn / T, dim=-1)

    # CNN: channel-mean of layer4 → [B,7,7] → bilinear upsample → [B,196]
    cnn_attn = teacher_spatial.mean(dim=1)               # [B, 7, 7]
    cnn_attn = F.interpolate(cnn_attn.unsqueeze(1), size=(14, 14),
                             mode='bilinear', align_corners=False).squeeze(1)
    cnn_attn = F.softmax(cnn_attn.view(B, -1) / T, dim=-1)

    return F.kl_div((vit_attn + 1e-8).log(), cnn_attn.detach(), reduction='batchmean')

def make_scheduler(opt, total_epochs, base_lr, min_lr=1e-6):
    def lr_fn(epoch):
        p = epoch / max(total_epochs - 1, 1)
        c = 0.5 * (1 + math.cos(math.pi * p))
        return (min_lr / base_lr) + (1 - min_lr / base_lr) * c
    return torch.optim.lr_scheduler.LambdaLR(opt, lr_fn)

results_p2 = {}

for DATA_SCALE in DATA_SCALES:
    print(f"\n{'='*65}\n  P2 | {DATA_SCALE}\n{'='*65}")
    CKPT = f"p2_{DATA_SCALE}.pth"

    n = int(len(full_trainset) * SCALE_MAP[DATA_SCALE])
    trainloader = torch.utils.data.DataLoader(
        torch.utils.data.Subset(full_trainset, all_indices[:n]),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    testloader = torch.utils.data.DataLoader(
        testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    print(f"  {n} images | {len(trainloader)} batches/epoch")

    teacher = torchvision.models.resnet18(weights=None)
    teacher.fc = nn.Linear(teacher.fc.in_features, 100)
    teacher.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    teacher = teacher.to(DEVICE).eval()

    student = timm.create_model('deit_tiny_distilled_patch16_224',
                                 pretrained=False, num_classes=100,
                                 drop_path_rate=DROP_PATH_RATE)
    student.set_distilled_training(True)
    student = student.to(DEVICE)

    proj_head = nn.Linear(192, 512).to(DEVICE)
    criterion_ce = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    # Hooks: semantic features + spatial features + QKV for attention
    cache = {}
    teacher.avgpool.register_forward_hook(
        lambda m, i, o: cache.update({'t_pool': o.view(o.size(0), -1)}))
    teacher.layer4.register_forward_hook(
        lambda m, i, o: cache.update({'t_spatial': o}))         # [B, 512, 7, 7]
    student.head_dist.register_forward_hook(
        lambda m, i, o: cache.update({'s_dist': i[0]}))
    student.blocks[-1].attn.qkv.register_forward_hook(
        lambda m, i, o: cache.update({'qkv': o}))               # [B, N, 3*C]

    params = list(student.parameters()) + list(proj_head.parameters())
    opt    = optim.AdamW(params, lr=LEARNING_RATE, weight_decay=0.05)
    sched  = make_scheduler(opt, EPOCHS, LEARNING_RATE)

    start_epoch = 0; best_acc = 0.0
    if os.path.exists(CKPT):
        ck = torch.load(CKPT)
        student.load_state_dict(ck['model']); proj_head.load_state_dict(ck['proj'])
        opt.load_state_dict(ck['opt']); sched.load_state_dict(ck['sched'])
        start_epoch = ck['epoch'] + 1; best_acc = ck['best_acc']
        print(f"  Resumed from epoch {start_epoch}")

    for epoch in range(start_epoch, EPOCHS):
        student.train(); proj_head.train()
        run_loss = 0.0
        sw = math.exp(-DECAY_RATE * epoch)
        aw = ATTN_WEIGHT * math.exp(-DECAY_RATE * epoch)    # attention weight also anneals

        for inputs, targets in trainloader:
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            opt.zero_grad()
            with torch.no_grad(): t_out = teacher(inputs)
            out = student(inputs)
            s_cls, s_dist = out if isinstance(out, tuple) else (out, out)

            attn_mod = student.blocks[-1].attn
            loss = (TASK_WEIGHT    * criterion_ce(s_cls, targets)
                  + DISTILL_WEIGHT * soft_kd_loss(s_dist, t_out)
                  + sw             * cosine_sem_loss(cache['t_pool'], cache['s_dist'], proj_head)
                  + aw             * attention_alignment_loss(
                                         cache['t_spatial'], cache['qkv'],
                                         attn_mod, inputs.size(0)))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step()
            run_loss += loss.item()

        sched.step()
        student.eval()
        correct = total = 0
        with torch.no_grad():
            for inputs, targets in testloader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                out = student(inputs)
                if isinstance(out, tuple): out = (out[0] + out[1]) / 2
                correct += out.max(1)[1].eq(targets).sum().item(); total += targets.size(0)
        val_acc = 100. * correct / total
        if val_acc > best_acc: best_acc = val_acc
        print(f"  Ep {epoch+1:2d} | Loss {run_loss/len(trainloader):.4f} | "
              f"Val {val_acc:.2f}% | SW {sw:.3f} | AW {aw:.3f} | LR {opt.param_groups[0]['lr']:.5f}")
        torch.save({'epoch': epoch, 'model': student.state_dict(), 'proj': proj_head.state_dict(),
                    'opt': opt.state_dict(), 'sched': sched.state_dict(), 'best_acc': best_acc}, CKPT)

    results_p2[DATA_SCALE] = best_acc
    print(f"  ✓ Best: {best_acc:.2f}%")

print(f"\n{'='*65}\nP2 SUMMARY\n{'='*65}")
for s, a in results_p2.items(): print(f"  {s:>5}: {a:.2f}%")


  P2 | 10%
  5000 images | 79 batches/epoch
  Ep  1 | Loss 8.4149 | Val 3.70% | SW 1.000 | AW 0.500 | LR 0.00078
  Ep  2 | Loss 7.5814 | Val 5.40% | SW 0.905 | AW 0.452 | LR 0.00071
  Ep  3 | Loss 7.0896 | Val 7.13% | SW 0.819 | AW 0.409 | LR 0.00060
  Ep  4 | Loss 6.7407 | Val 7.14% | SW 0.741 | AW 0.370 | LR 0.00047
  Ep  5 | Loss 6.4828 | Val 8.87% | SW 0.670 | AW 0.335 | LR 0.00033
  Ep  6 | Loss 6.2329 | Val 10.03% | SW 0.607 | AW 0.303 | LR 0.00020
  Ep  7 | Loss 6.0193 | Val 10.79% | SW 0.549 | AW 0.274 | LR 0.00009
  Ep  8 | Loss 5.8339 | Val 11.45% | SW 0.497 | AW 0.248 | LR 0.00003
  Ep  9 | Loss 5.7318 | Val 11.78% | SW 0.449 | AW 0.225 | LR 0.00000
  Ep 10 | Loss 5.7177 | Val 11.78% | SW 0.407 | AW 0.203 | LR 0.00003
  ✓ Best: 11.78%

  P2 | 25%
  12500 images | 196 batches/epoch
  Ep  1 | Loss 7.8434 | Val 5.70% | SW 1.000 | AW 0.500 | LR 0.00078
  Ep  2 | Loss 6.8534 | Val 9.43% | SW 0.905 | AW 0.452 | LR 0.00071
  Ep  3 | Loss 6.3562 | Val 11.00% | SW 0.819 | AW 0.409 |